# MakFleet ST-GNN Training — Google Colab
**BIS 3205 | Makerere University**

This notebook trains the MakFleet Spatio-Temporal Graph Neural Network on GPU.

## Before running:
1. Upload these files to your Google Drive under a folder called `MakFleet/data/`:
   - `pyg_train.pt`
   - `pyg_val.pt`
   - `pyg_test_temporal.pt`
   - `pyg_test_ood.pt`
2. Set Runtime → Change runtime type → **A100 GPU**
3. Run all cells top to bottom

In [1]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    print(f'VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU')

KeyboardInterrupt: 

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['pip', 'install', '-q',
     'torch_geometric',
     'torch_scatter', 'torch_sparse',
     '--find-links', f'https://data.pyg.org/whl/torch-{torch.__version__}.html'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else 'torch_geometric installed')
print(result.stderr[-300:] if result.stderr else '')

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/MakFleet/data'
OUT_DIR  = '/content/drive/MyDrive/MakFleet'

# Verify files exist
for fname in ['pyg_train.pt', 'pyg_val.pt', 'pyg_test_temporal.pt', 'pyg_test_ood.pt']:
    path = os.path.join(DATA_DIR, fname)
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1e6 if exists else 0
    print(f'  {fname}: {"OK" if exists else "MISSING"} ({size:.1f} MB)')

In [ ]:
# ── Cell 4: Load dataset ───────────────────────────────────────────────────
import pickle

def load_split(name):
    with open(f'{DATA_DIR}/pyg_{name}.pt', 'rb') as f:
        return pickle.load(f)

train_s       = load_split('train')
val_s         = load_split('val')
test_temporal = load_split('test_temporal')
test_ood      = load_split('test_ood')

print(f'Train:         {len(train_s):,} samples')
print(f'Val:           {len(val_s):,} samples')
print(f'Test temporal: {len(test_temporal):,} samples')
print(f'Test OOD:      {len(test_ood):,} samples')

# Quick sanity check
sample = train_s[0]
print(f'\nSample keys:     {list(sample.keys())}')
print(f'Snapshots per sample: {len(sample["snapshots"])}')
print(f'Nodes per snapshot:   {sample["snapshots"][0].x.shape[0]}')
print(f'Node features:        {sample["snapshots"][0].x.shape[1]}')
print(f'Positive rate:        {sum(s["label"] for s in train_s)/len(train_s)*100:.1f}%')

In [ ]:
# ── Cell 5: Model definition ───────────────────────────────────────────────
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')

class STGNNModel(nn.Module):
    """
    Spatio-Temporal GNN: GCN spatial encoder + LSTM temporal aggregator.
    Equivalent to A3T-GCN but implemented from primitives.
    """
    def __init__(self, in_channels=6, hidden=64, lstm_layers=2):
        super().__init__()
        self.gcn1 = GCNConv(in_channels, hidden)
        self.gcn2 = GCNConv(hidden, hidden)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(
            input_size=hidden, hidden_size=hidden,
            num_layers=lstm_layers, batch_first=True
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
            nn.Sigmoid(),
        )

    def encode_snapshot(self, x, edge_index):
        h = self.relu(self.gcn1(x, edge_index))
        h = self.relu(self.gcn2(h, edge_index))
        return h

    def forward(self, snapshot_list):
        if not snapshot_list:
            return torch.zeros(1, device=DEVICE)
        encoded = []
        for snap in snapshot_list:
            x  = snap.x.to(DEVICE)
            ei = snap.edge_index.to(DEVICE)
            if ei.numel() == 0:
                ei = torch.zeros((2, 0), dtype=torch.long, device=DEVICE)
            encoded.append(self.encode_snapshot(x, ei))
        H = torch.stack(encoded, dim=1)       # (N, T, hidden)
        out, _ = self.lstm(H)                  # (N, T, hidden)
        h_final = out[:, -1, :]               # (N, hidden)
        return self.head(h_final).squeeze(-1)  # (N,)

model = STGNNModel(in_channels=6, hidden=64, lstm_layers=2).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')
print(model)

In [ ]:
# ── Cell 6: Training utilities ─────────────────────────────────────────────
import statistics
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

POS_WEIGHT = torch.tensor(19.0, device=DEVICE)

def weighted_bce(probs, targets, pos_weight=POS_WEIGHT):
    weights = torch.where(targets > 0, pos_weight, torch.tensor(1.0, device=DEVICE))
    eps = 1e-7
    loss = -(weights * (targets * torch.log(probs + eps) +
                        (1 - targets) * torch.log(1 - probs + eps)))
    return loss.mean()

def evaluate(model, samples, split_name):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for sample in samples:
            snaps = sample['snapshots']
            if not snaps:
                continue
            probs = model(snaps)
            all_probs.append(float(probs.max().item()))
            all_labels.append(sample['label'])
    preds = [1 if p > 0.5 else 0 for p in all_probs]
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0
    metrics = {
        'auc':       round(auc, 4),
        'f1':        round(f1_score(all_labels, preds, zero_division=0), 4),
        'precision': round(precision_score(all_labels, preds, zero_division=0), 4),
        'recall':    round(recall_score(all_labels, preds, zero_division=0), 4),
    }
    print(f'  [{split_name}] AUC={metrics["auc"]} F1={metrics["f1"]} '
          f'P={metrics["precision"]} R={metrics["recall"]}')
    return metrics

print('Utilities ready')

In [ ]:
# ── Cell 7: Training loop ──────────────────────────────────────────────────
import random
import time
import json
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS    = 50
LR        = 1e-3
PATIENCE  = 10   # evaluation steps (each = 5 epochs)

optimizer = Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_auc   = 0.0
no_improve = 0
history    = []
checkpoint_path = f'{OUT_DIR}/stgnn_checkpoint_colab.pt'

print(f'Training ST-GNN for {EPOCHS} epochs on {DEVICE}')
print(f'Train: {len(train_s):,} samples | Val: {len(val_s):,} samples')
print('=' * 60)

t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    random.shuffle(train_s)

    for sample in train_s:
        snaps = sample['snapshots']
        label = sample['label']
        if not snaps:
            continue
        optimizer.zero_grad()
        probs   = model(snaps)
        targets = torch.full((probs.size(0),), label,
                             dtype=torch.float, device=DEVICE)
        loss = weighted_bce(probs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / max(len(train_s), 1)

    if epoch % 5 == 0 or epoch == EPOCHS:
        elapsed = time.time() - t_start
        print(f'\nEpoch {epoch:3d}/{EPOCHS} | loss={avg_loss:.4f} | elapsed={elapsed:.0f}s')
        val_m = evaluate(model, val_s, 'val')
        val_auc = val_m['auc']
        history.append({'epoch': epoch, 'loss': avg_loss, **val_m})

        if val_auc > best_auc:
            best_auc   = val_auc
            no_improve = 0
            torch.save(model.state_dict(), checkpoint_path)
            print(f'  ✓ Checkpoint saved (AUC={best_auc:.4f})')
        else:
            no_improve += 1
            print(f'  No improvement ({no_improve}/{PATIENCE})')
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

print('\n' + '=' * 60)
print(f'Training complete. Best val AUC: {best_auc:.4f}')

In [ ]:
# ── Cell 8: Full evaluation ────────────────────────────────────────────────
print('Loading best checkpoint for evaluation...')
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))

print('\n=== FINAL EVALUATION ===')
val_m      = evaluate(model, val_s,          'Validation')
temporal_m = evaluate(model, test_temporal,  'Temporal OOD (Semester 2)')
ood_m      = evaluate(model, test_ood,       'Spatial OOD (Eastern campus)')

results = {
    'best_val_auc':  best_auc,
    'validation':    val_m,
    'temporal_ood':  temporal_m,
    'spatial_ood':   ood_m,
    'epochs_trained': len(history) * 5,
    'history':        history,
}

results_path = f'{OUT_DIR}/eval_results_colab.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved to Google Drive: {results_path}')

In [ ]:
# ── Cell 9: Training curve plot ────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs_log = [h['epoch'] for h in history]
loss_log   = [h['loss']  for h in history]
auc_log    = [h['auc']   for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_log, loss_log, 'o-', color='#e74c3c', linewidth=2)
axes[0].set_title('Training Loss per Evaluation Step', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Weighted BCE Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_log, auc_log, 'o-', color='#2ecc71', linewidth=2)
axes[1].axhline(max(auc_log), color='red', linestyle='--',
                label=f'Best AUC = {max(auc_log):.4f}')
axes[1].set_title('Validation AUC-ROC per Evaluation Step', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC-ROC')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = f'{OUT_DIR}/training_curve.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Training curve saved: {plot_path}')

In [ ]:
# ── Cell 10: Results summary table ─────────────────────────────────────────
import pandas as pd

summary = pd.DataFrame([
    ['Validation (Sem1 held-out)',       val_m['auc'],      val_m['f1'],      val_m['precision'],      val_m['recall']],
    ['Temporal OOD (Semester 2)',        temporal_m['auc'], temporal_m['f1'], temporal_m['precision'], temporal_m['recall']],
    ['Spatial OOD (Eastern campus)',     ood_m['auc'],      ood_m['f1'],      ood_m['precision'],      ood_m['recall']],
], columns=['Split', 'AUC-ROC', 'F1', 'Precision', 'Recall'])

print('\n=== ST-GNN RESULTS TABLE ===')
print(summary.to_string(index=False))
print(f'\nBest checkpoint: {checkpoint_path}')
print('\nDownload stgnn_checkpoint_colab.pt from Google Drive')
print('and place it at: data/stgnn_checkpoint.pt on your local machine')
print('The dashboard will automatically load it as the neural scorer.')